# Atividade SARIMAX — Pipeline completa

Este notebook resolve as Tarefas 1, 2 e 3 da Aula 8.

**Fluxo**

1. Carrega e valida a série semanal.
2. Analisa autocovariância, autocorrelação, ACF/PACF, ADF e KPSS.
3. Separa as últimas 42 semanas para teste, sem embaralhamento.
4. Executa a Tarefa 1 com o benchmark da aula.
5. Executa a Tarefa 2 com cenários futuros de feriado.
6. Busca exaustivamente os parâmetros dos Base Models, SARIMA e SARIMAX.
7. Testa cada sazonalidade inteira de `s=0` até `s=100`.
8. Imprime cada configuração testada e as tabelas completas ordenadas.
9. Compara os vencedores, diagnostica resíduos e apresenta rankings.

> **Execução completa:** é o comportamento padrão e pode demorar bastante. Os resultados são persistidos em JSONL após cada candidato e a busca pode ser retomada.
>
> **Teste técnico reduzido:** definir a variável de ambiente `SARIMAX_SMOKE_TEST=1` antes de executar. Esse modo existe somente para verificar a integridade do código.


## 1. Bibliotecas e configurações da análise


In [ ]:
import itertools
import json
import os
import platform
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)

DATA_PATH = Path("dados_aula08_sarimax.xlsx")
OUTPUT_DIR = Path("resultados_sarimax")
OUTPUT_DIR.mkdir(exist_ok=True)

TEST_SIZE = 42
N_FOLDS = 3
VALIDATION_HORIZON = 13
ALPHA = 0.05
MAXITER = 100
PRINT_EVERY_CANDIDATE = True

SMOKE_TEST = False
RUN_LABEL = "smoke" if SMOKE_TEST else "full"
CHECKPOINT_DIR = OUTPUT_DIR / f"checkpoints_{RUN_LABEL}"
CHECKPOINT_DIR.mkdir(exist_ok=True)

if SMOKE_TEST:
    P_VALUES = [0]
    D_VALUES = [1]
    Q_VALUES = [0]
    SP_VALUES = [0, 1]
    SD_VALUES = [0]
    SQ_VALUES = [0]
    S_VALUES = [0, 1, 4]
    TREND_VALUES = ["n"]
else:
    P_VALUES = [0, 1, 2, 3]
    D_VALUES = [0, 1]
    Q_VALUES = [0, 1, 2, 3]
    SP_VALUES = [0, 1, 2]
    SD_VALUES = [0, 1]
    SQ_VALUES = [0, 1, 2]
    S_VALUES = list(range(0, 101))
    TREND_VALUES = ["n", "c", "t"]

EXOG_SETS = {
    "temperatura": ["temperatura"],
    "feriado": ["feriado"],
    "temperatura_feriado": ["temperatura", "feriado"],
    "temperatura_feriado_dezembro": ["temperatura", "feriado", "dezembro"],
}

print(f"Modo: {'TESTE REDUZIDO' if SMOKE_TEST else 'BUSCA COMPLETA'}")
print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("Configuração carregada.")


## 2. Conhecendo e preparando os dados


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {DATA_PATH.resolve()}")

df_original = pd.read_excel(DATA_PATH, sheet_name="Sheet1")
colunas_esperadas = ["data", "vendas", "temperatura", "feriado"]

if list(df_original.columns) != colunas_esperadas:
    raise ValueError(f"Colunas diferentes do esperado: {list(df_original.columns)}")

df_original["data"] = pd.to_datetime(df_original["data"], errors="raise")
df_original = df_original.sort_values("data").reset_index(drop=True)

if df_original["data"].duplicated().any():
    raise ValueError("Foram encontradas datas duplicadas.")
if df_original.isna().any().any():
    raise ValueError("Foram encontrados valores ausentes.")
if not set(df_original["feriado"].unique()).issubset({0, 1}):
    raise ValueError("A coluna feriado deve conter apenas 0 e 1.")

diferencas = df_original["data"].diff().dropna()
if not (diferencas == pd.Timedelta(days=7)).all():
    raise ValueError("A série não possui frequência semanal regular de 7 dias.")

df = df_original.copy()
df["dezembro"] = (df["data"].dt.month == 12).astype(int)
df = df.set_index("data").asfreq("W-MON")

print("Dimensão:", df.shape)
print("Período:", df.index.min().date(), "até", df.index.max().date())
print("Frequência:", df.index.freqstr)
print("Nulos por coluna:")
print(df.isna().sum())
display(df.head())
display(df.describe().T)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
axes[0].plot(df.index, df["vendas"], color="#173f73")
axes[0].set_title("Vendas semanais")
axes[0].set_ylabel("Vendas")
axes[1].plot(df.index, df["temperatura"], color="#dc5a3a")
axes[1].set_title("Temperatura")
axes[1].set_ylabel("°C")
axes[2].stem(df.index, df["feriado"], linefmt="#2a9d8f", markerfmt="o", basefmt=" ")
axes[2].set_title("Feriados")
axes[2].set_ylabel("0/1")
plt.tight_layout()
plt.show()

correlacoes = df[["vendas", "temperatura", "feriado", "dezembro"]].corr()
display(correlacoes)

exog_vif = df[["temperatura", "feriado", "dezembro"]].astype(float)
vif = pd.DataFrame({
    "variavel": exog_vif.columns,
    "VIF": [variance_inflation_factor(exog_vif.values, i) for i in range(exog_vif.shape[1])],
})
display(vif)


## 3. Análise da dependência temporal


In [ ]:
def autocovariance(series, lag):
    values = np.asarray(series, dtype=float)
    n = len(values)
    mean = values.mean()
    if lag < 0 or lag >= n:
        raise ValueError("Lag fora do intervalo da série.")
    return np.sum((values[lag:] - mean) * (values[: n - lag] - mean)) / n

def autocorrelation(series, lag):
    return autocovariance(series, lag) / autocovariance(series, 0)

serie_vendas = df["vendas"]
max_lag = min(60, len(serie_vendas) // 2 - 1)
lags = np.arange(max_lag + 1)
autocov_manual = [autocovariance(serie_vendas, lag) for lag in lags]
autocorr_manual = [autocorrelation(serie_vendas, lag) for lag in lags]

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(lags, autocov_manual, marker="o", markersize=3)
axes[0].set_title("Autocovariância manual")
axes[0].set_xlabel("Lag")
axes[1].plot(lags, autocorr_manual, marker="o", markersize=3, color="#e76f51")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Autocorrelação manual")
axes[1].set_xlabel("Lag")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
plot_acf(serie_vendas, lags=max_lag, ax=axes[0], title="ACF — statsmodels")
plot_pacf(serie_vendas, lags=max_lag, ax=axes[1], method="ywm", title="PACF — statsmodels")
plt.tight_layout()
plt.show()


## 4. Verificação da estacionariedade


In [ ]:
def adf_test(series, alpha=0.05):
    result = adfuller(pd.Series(series).dropna(), autolag="AIC")
    output = {
        "estatistica": result[0], "p_valor": result[1], "lags": result[2],
        "observacoes": result[3], "valores_criticos": result[4],
        "estacionaria": result[1] < alpha,
    }
    print("ADF")
    print(f"  Estatística: {output['estatistica']:.6f}")
    print(f"  p-valor: {output['p_valor']:.6f}")
    print("  Valores críticos:", output["valores_criticos"])
    return output

def kpss_test(series, alpha=0.05):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = kpss(pd.Series(series).dropna(), regression="c", nlags="auto")
    output = {
        "estatistica": result[0], "p_valor": result[1], "lags": result[2],
        "valores_criticos": result[3], "estacionaria": result[1] > alpha,
    }
    print("KPSS")
    print(f"  Estatística: {output['estatistica']:.6f}")
    print(f"  p-valor: {output['p_valor']:.6f}")
    print("  Valores críticos:", output["valores_criticos"])
    return output

resultado_adf = adf_test(serie_vendas, ALPHA)
resultado_kpss = kpss_test(serie_vendas, ALPHA)
estacionaria = resultado_adf["estacionaria"] and resultado_kpss["estacionaria"]
print("SÉRIE ESTACIONÁRIA" if estacionaria else "SÉRIE NÃO ESTACIONÁRIA")


## 5. Separação entre treino e teste


In [ ]:
df_train = df.iloc[:-TEST_SIZE].copy()
df_test = df.iloc[-TEST_SIZE:].copy()
if len(df_test) != TEST_SIZE:
    raise AssertionError("Horizonte de teste incorreto.")

print("Treino:", df_train.index.min().date(), "até", df_train.index.max().date(), len(df_train))
print("Teste :", df_test.index.min().date(), "até", df_test.index.max().date(), len(df_test))

plt.figure(figsize=(14, 5))
plt.plot(df.index, df["vendas"], label="Série")
plt.axvline(df_test.index[0], color="red", linestyle="--", label="Início do teste")
plt.axvspan(df_test.index[0], df_test.index[-1], color="red", alpha=0.08)
plt.title("Separação temporal treino/teste")
plt.ylabel("Vendas")
plt.legend()
plt.tight_layout()
plt.show()

def mape_safe(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else np.nan

def metrics(y_true, y_pred):
    return {"mae": float(mean_absolute_error(y_true, y_pred)), "mape": float(mape_safe(y_true, y_pred))}

def scale_exog(train_exog, other_exog):
    train_scaled = train_exog.astype(float).copy()
    other_scaled = other_exog.astype(float).copy()
    continuous = [col for col in ["temperatura"] if col in train_scaled.columns]
    scaler = None
    if continuous:
        scaler = StandardScaler()
        train_scaled.loc[:, continuous] = scaler.fit_transform(train_scaled[continuous])
        other_scaled.loc[:, continuous] = scaler.transform(other_scaled[continuous])
    return train_scaled, other_scaled, scaler

def fit_sarimax_model(y, exog, order, seasonal_order, trend):
    started = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model = SARIMAX(
            endog=y, exog=exog, order=tuple(order), seasonal_order=tuple(seasonal_order),
            trend=trend, enforce_stationarity=False, enforce_invertibility=False,
        )
        result = model.fit(disp=False, maxiter=MAXITER)
    elapsed = time.perf_counter() - started
    converged = bool(result.mle_retvals.get("converged", True))
    warning_text = " | ".join(str(w.message) for w in caught[:3])
    return result, converged, elapsed, warning_text

def fit_and_forecast(train, future, order, seasonal_order, trend="n", exog_cols=None):
    exog_cols = list(exog_cols or [])
    if exog_cols:
        x_train, x_future, scaler = scale_exog(train[exog_cols], future[exog_cols])
    else:
        x_train = x_future = scaler = None
    result, converged, elapsed, warning_text = fit_sarimax_model(
        train["vendas"], x_train, order, seasonal_order, trend
    )
    forecast = result.get_forecast(steps=len(future), exog=x_future).predicted_mean
    forecast.index = future.index
    return {"result": result, "forecast": forecast, "converged": converged, "elapsed": elapsed, "warning": warning_text, "scaler": scaler}


## 6. Tarefa 1 — Efeito do mês de dezembro


In [ ]:
BENCHMARK_ORDER = (0, 1, 2)
BENCHMARK_SEASONAL = (0, 1, 1, 52)
task1_specs = [
    ("SARIMA_sem_exog", []),
    ("SARIMAX_temperatura_feriado", ["temperatura", "feriado"]),
    ("SARIMAX_temperatura_feriado_dezembro", ["temperatura", "feriado", "dezembro"]),
]
task1_rows, task1_forecasts = [], {}
for model_name, exog_cols in task1_specs:
    fitted = fit_and_forecast(df_train, df_test, BENCHMARK_ORDER, BENCHMARK_SEASONAL, "n", exog_cols)
    score = metrics(df_test["vendas"], fitted["forecast"])
    task1_rows.append({
        "modelo": model_name, "order": BENCHMARK_ORDER, "seasonal_order": BENCHMARK_SEASONAL,
        "exogenas": ", ".join(exog_cols) if exog_cols else "nenhuma",
        "BIC": fitted["result"].bic, "MAE": score["mae"], "MAPE": score["mape"],
        "convergiu": fitted["converged"],
    })
    task1_forecasts[model_name] = fitted["forecast"]

task1_benchmark = pd.DataFrame(task1_rows).sort_values(["MAE", "BIC"])
display(task1_benchmark)
plt.figure(figsize=(14, 6))
plt.plot(df_test.index, df_test["vendas"], color="black", linewidth=2, label="Real")
for name, forecast in task1_forecasts.items():
    plt.plot(df_test.index, forecast, label=name)
plt.title("Tarefa 1 — comparação dos três modelos benchmark")
plt.ylabel("Vendas")
plt.legend()
plt.tight_layout()
plt.show()


## 7. Tarefa 2 — Previsões com e sem feriado


In [ ]:
future_index = pd.date_range(start=df.index[-1] + pd.Timedelta(weeks=1), periods=4, freq="W-MON")
future_no_holiday = pd.DataFrame({"temperatura": 22.0, "feriado": 0}, index=future_index)
future_holiday = pd.DataFrame({"temperatura": 22.0, "feriado": 1}, index=future_index)

x_full = df[["temperatura", "feriado"]].astype(float).copy()
x_no_holiday = future_no_holiday.astype(float).copy()
x_holiday = future_holiday.astype(float).copy()
scenario_scaler = StandardScaler()
x_full.loc[:, ["temperatura"]] = scenario_scaler.fit_transform(x_full[["temperatura"]])
x_no_holiday.loc[:, ["temperatura"]] = scenario_scaler.transform(x_no_holiday[["temperatura"]])
x_holiday.loc[:, ["temperatura"]] = scenario_scaler.transform(x_holiday[["temperatura"]])

scenario_model, _, _, _ = fit_sarimax_model(df["vendas"], x_full, BENCHMARK_ORDER, BENCHMARK_SEASONAL, "n")
forecast_no_holiday = scenario_model.get_forecast(steps=4, exog=x_no_holiday).predicted_mean
forecast_holiday = scenario_model.get_forecast(steps=4, exog=x_holiday).predicted_mean
forecast_no_holiday.index = future_index
forecast_holiday.index = future_index

task2_result = pd.DataFrame({"sem_feriado": forecast_no_holiday, "com_feriado": forecast_holiday})
task2_result["diferenca"] = task2_result["com_feriado"] - task2_result["sem_feriado"]
display(task2_result)
print(f"Diferença média: {task2_result['diferenca'].mean():.2f} unidades")
task2_result[["sem_feriado", "com_feriado"]].plot(kind="bar", figsize=(11, 5), color=["#244a7c", "#e15d3f"])
plt.title("Tarefa 2 — impacto de feriados nas próximas quatro semanas")
plt.xlabel("Semana")
plt.ylabel("Vendas previstas")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 8. Tarefa 3 — Pipeline completa

### 8.1 Funções auxiliares para testar e comparar os modelos


In [ ]:
def validation_folds(n, n_folds=N_FOLDS, horizon=VALIDATION_HORIZON):
    initial = n - n_folds * horizon
    if initial <= 0:
        raise ValueError("Treino insuficiente para os folds solicitados.")
    return [(np.arange(0, initial + fold * horizon), np.arange(initial + fold * horizon, initial + (fold + 1) * horizon)) for fold in range(n_folds)]

FOLDS = validation_folds(len(df_train))
print("Folds temporais:")
for i, (train_idx, val_idx) in enumerate(FOLDS, 1):
    print(i, df_train.index[train_idx[0]].date(), "→", df_train.index[train_idx[-1]].date(), "| validação:", df_train.index[val_idx[0]].date(), "→", df_train.index[val_idx[-1]].date())

def json_default(value):
    if isinstance(value, np.integer): return int(value)
    if isinstance(value, np.floating): return float(value)
    if isinstance(value, np.bool_): return bool(value)
    if isinstance(value, (tuple, set)): return list(value)
    if pd.isna(value): return None
    raise TypeError(f"Tipo não serializável: {type(value)}")

class ResultStore:
    def __init__(self, name):
        self.path = CHECKPOINT_DIR / f"{name}.jsonl"
        self.records, self.seen = [], set()
        if self.path.exists():
            with self.path.open("r", encoding="utf-8") as stream:
                for line in stream:
                    try:
                        record = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    self.records.append(record)
                    self.seen.add(record["candidate_key"])
            print(f"Retomando {name}: {len(self.records)} candidatos carregados.")
    def add(self, record):
        with self.path.open("a", encoding="utf-8") as stream:
            stream.write(json.dumps(record, ensure_ascii=False, default=json_default) + "\n")
        self.records.append(record)
        self.seen.add(record["candidate_key"])
    def frame(self):
        return pd.DataFrame(self.records)

def candidate_key(payload):
    return json.dumps(payload, sort_keys=True, ensure_ascii=False)

def print_candidate(record, current, total):
    if PRINT_EVERY_CANDIDATE:
        print(
            f"[{current:>7}/{total}] {record['family']:<10} status={record['status']:<15} "
            f"order={record.get('order')} seasonal={record.get('seasonal_order')} "
            f"exog={record.get('exog_key', 'nenhuma')} trend={record.get('trend', '-')} "
            f"MAE_val={record.get('val_mae')} BIC={record.get('bic')} "
            f"tempo={record.get('elapsed_seconds')}s erro={record.get('error', '')}"
        )

def sort_results(frame):
    if frame.empty: return frame
    ordered = frame.copy()
    ordered["_ok"] = ordered["status"].eq("ok")
    ordered["_mae"] = pd.to_numeric(ordered.get("val_mae"), errors="coerce").fillna(np.inf)
    ordered["_bic"] = pd.to_numeric(ordered.get("bic"), errors="coerce").fillna(np.inf)
    return ordered.sort_values(["_ok", "_mae", "_bic"], ascending=[False, True, True]).drop(columns=["_ok", "_mae", "_bic"]).reset_index(drop=True)

def print_full_ranking(frame, title):
    ranking = sort_results(frame)
    if not ranking.empty:
        ranking = ranking.copy()
        ranking.insert(0, "posicao", np.arange(1, len(ranking) + 1))
        ranking.insert(1, "destaque", ["MELHOR MODELO"] + [""] * (len(ranking) - 1))
    print("\n" + "=" * 120)
    print(title)
    print("=" * 120)
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 260, "display.max_colwidth", 100):
        print(ranking.to_string(index=False))
    return ranking

def invalid_record(family, payload, error):
    return {
        "candidate_key": candidate_key(payload), "family": family, "status": "invalido",
        "order": payload.get("order"), "seasonal_order": payload.get("seasonal_order"),
        "s": payload.get("s"), "trend": payload.get("trend"),
        "exog_key": payload.get("exog_key", "nenhuma"), "exog_cols": payload.get("exog_cols", []),
        "val_mae": None, "val_mae_std": None, "val_mape": None, "bic": None, "aic": None,
        "converged": False, "elapsed_seconds": 0.0, "warning": "", "error": error,
    }


### 8.2 Teste e comparação dos Base Models


In [ ]:
def base_candidates():
    yield {"model": "media_historica", "window": None}
    yield {"model": "mediana_historica", "window": None}
    for lag in [1, 2, 4, 13, 26, 52]: yield {"model": "naive", "lag": lag}
    for window in [4, 8, 13, 26, 52]:
        yield {"model": "media_movel", "window": window}
        yield {"model": "mediana_movel", "window": window}
    for lookback in [None, 13, 26, 52, 104]: yield {"model": "drift", "lookback": lookback}
    for s in S_VALUES: yield {"model": "seasonal_naive", "s": s}

def base_forecast(y_train, horizon, params):
    values, model = np.asarray(y_train, dtype=float), params["model"]
    if model == "media_historica": return np.repeat(values.mean(), horizon)
    if model == "mediana_historica": return np.repeat(np.median(values), horizon)
    if model == "media_movel":
        window = params["window"]
        if len(values) < window: raise ValueError("Histórico insuficiente para a janela.")
        return np.repeat(values[-window:].mean(), horizon)
    if model == "mediana_movel":
        window = params["window"]
        if len(values) < window: raise ValueError("Histórico insuficiente para a janela.")
        return np.repeat(np.median(values[-window:]), horizon)
    if model == "naive":
        lag = params["lag"]
        if len(values) < lag: raise ValueError("Histórico insuficiente para o lag.")
        pattern = values[-lag:]
        return np.array([pattern[i % lag] for i in range(horizon)])
    if model == "seasonal_naive":
        s = params["s"]
        if s == 0: raise ValueError("s=0 representa ausência de sazonalidade.")
        if s == 1: raise ValueError("s=1 não é sazonalidade aplicável.")
        if len(values) < s: raise ValueError("Histórico insuficiente para a sazonalidade.")
        pattern = values[-s:]
        return np.array([pattern[i % s] for i in range(horizon)])
    if model == "drift":
        lookback = params["lookback"]
        selected = values if lookback is None else values[-lookback:]
        if len(selected) < 2: raise ValueError("Histórico insuficiente para drift.")
        slope = (selected[-1] - selected[0]) / (len(selected) - 1)
        return selected[-1] + slope * np.arange(1, horizon + 1)
    raise ValueError(f"Base Model desconhecido: {model}")

def search_base_models():
    candidates, store = list(base_candidates()), ResultStore("base_models")
    total, started_all = len(candidates), time.perf_counter()
    for current, params in enumerate(candidates, 1):
        payload = {"family": "BASE", **params}
        key = candidate_key(payload)
        if key in store.seen: continue
        started = time.perf_counter()
        try:
            fold_mae, fold_mape = [], []
            for train_idx, val_idx in FOLDS:
                pred = base_forecast(df_train["vendas"].iloc[train_idx], len(val_idx), params)
                score = metrics(df_train["vendas"].iloc[val_idx], pred)
                fold_mae.append(score["mae"]); fold_mape.append(score["mape"])
            record = {
                "candidate_key": key, "family": "BASE", "status": "ok", "model": params["model"],
                "params": params, "order": None, "seasonal_order": None, "s": params.get("s"),
                "trend": None, "exog_key": "nenhuma", "exog_cols": [],
                "val_mae": float(np.mean(fold_mae)), "val_mae_std": float(np.std(fold_mae)),
                "val_mape": float(np.mean(fold_mape)), "bic": None, "aic": None,
                "converged": True, "elapsed_seconds": round(time.perf_counter() - started, 4),
                "warning": "", "error": "",
            }
        except Exception as exc:
            record = invalid_record("BASE", payload, str(exc))
            record.update({"model": params["model"], "params": params, "elapsed_seconds": round(time.perf_counter() - started, 4)})
        store.add(record); print_candidate(record, current, total)
    frame = store.frame()
    print(f"Tempo total Base Models: {time.perf_counter() - started_all:.2f}s")
    print_full_ranking(frame, "RANKING COMPLETO — BASE MODELS")
    return frame


### 8.3 Teste dos parâmetros de SARIMA, SARIMAX e sazonalidade


In [ ]:
def seasonal_combinations():
    for s in S_VALUES:
        if s == 0:
            yield (0, 0, 0, 0)
        else:
            for sp, sd, sq in itertools.product(SP_VALUES, SD_VALUES, SQ_VALUES):
                yield (sp, sd, sq, s)

def statistical_candidate_count(exog_sets_count=1, trend_count=1):
    order_count = len(P_VALUES) * len(D_VALUES) * len(Q_VALUES)
    seasonal_count = sum(1 for _ in seasonal_combinations())
    return order_count * seasonal_count * exog_sets_count * trend_count

def statistical_candidates(family):
    exog_items, trends = ([("nenhuma", [])], ["n"]) if family == "SARIMA" else (list(EXOG_SETS.items()), TREND_VALUES)
    for p, d, q in itertools.product(P_VALUES, D_VALUES, Q_VALUES):
        order = (p, d, q)
        for seasonal_order in seasonal_combinations():
            for exog_key, exog_cols in exog_items:
                for trend in trends:
                    yield {
                        "family": family, "order": order, "seasonal_order": seasonal_order,
                        "s": seasonal_order[3], "exog_key": exog_key,
                        "exog_cols": exog_cols, "trend": trend,
                    }

def evaluate_statistical_candidate(payload):
    family, order = payload["family"], tuple(payload["order"])
    seasonal_order, s = tuple(payload["seasonal_order"]), payload["s"]
    exog_cols, trend = list(payload["exog_cols"]), payload["trend"]
    if s == 1:
        return invalid_record(family, payload, "s=1 auditado, mas não representa periodicidade sazonal válida no statsmodels.")
    started, fold_mae, fold_mape, convergences, warning_messages = time.perf_counter(), [], [], [], []
    try:
        for train_idx, val_idx in FOLDS:
            fold_train, fold_val = df_train.iloc[train_idx], df_train.iloc[val_idx]
            if exog_cols: x_train, x_val, _ = scale_exog(fold_train[exog_cols], fold_val[exog_cols])
            else: x_train = x_val = None
            result, converged, _, warning_text = fit_sarimax_model(fold_train["vendas"], x_train, order, seasonal_order, trend)
            pred = result.get_forecast(steps=len(fold_val), exog=x_val).predicted_mean
            score = metrics(fold_val["vendas"], pred)
            fold_mae.append(score["mae"]); fold_mape.append(score["mape"]); convergences.append(converged)
            if warning_text: warning_messages.append(warning_text)
        if exog_cols:
            x_full_train = df_train[exog_cols].astype(float).copy()
            continuous = [c for c in ["temperatura"] if c in exog_cols]
            if continuous:
                scaler = StandardScaler()
                x_full_train.loc[:, continuous] = scaler.fit_transform(x_full_train[continuous])
        else: x_full_train = None
        full_result, full_converged, _, full_warning = fit_sarimax_model(df_train["vendas"], x_full_train, order, seasonal_order, trend)
        if full_warning: warning_messages.append(full_warning)
        all_converged = bool(all(convergences) and full_converged)
        return {
            "candidate_key": candidate_key(payload), "family": family,
            "status": "ok" if all_converged else "nao_convergiu",
            "order": list(order), "seasonal_order": list(seasonal_order), "s": s,
            "trend": trend, "exog_key": payload["exog_key"], "exog_cols": exog_cols,
            "val_mae": float(np.mean(fold_mae)), "val_mae_std": float(np.std(fold_mae)),
            "val_mape": float(np.mean(fold_mape)), "bic": float(full_result.bic),
            "aic": float(full_result.aic), "converged": all_converged,
            "elapsed_seconds": round(time.perf_counter() - started, 4),
            "warning": " | ".join(warning_messages[:3]), "error": "",
        }
    except Exception as exc:
        record = invalid_record(family, payload, str(exc))
        record["status"] = "erro"
        record["elapsed_seconds"] = round(time.perf_counter() - started, 4)
        return record

def search_statistical_models(family):
    if family == "SARIMA":
        total, store_name = statistical_candidate_count(), "sarima"
    else:
        total, store_name = statistical_candidate_count(len(EXOG_SETS), len(TREND_VALUES)), "sarimax"
    print(f"\n{family}: total esperado de candidatos = {total:,}")
    store, started_all = ResultStore(store_name), time.perf_counter()
    for current, payload in enumerate(statistical_candidates(family), 1):
        if candidate_key(payload) in store.seen: continue
        record = evaluate_statistical_candidate(payload)
        store.add(record); print_candidate(record, current, total)
    frame = store.frame()
    successes, failures = int(frame["status"].eq("ok").sum()), int((~frame["status"].eq("ok")).sum())
    print(f"{family}: esperado={total:,}, registrado={len(frame):,}, sucessos={successes:,}, falhas/inválidos={failures:,}")
    if len(frame) != total: raise AssertionError(f"Busca incompleta de {family}: {len(frame)} de {total}.")
    print(f"Tempo total {family}: {time.perf_counter() - started_all:.2f}s")
    print_full_ranking(frame, f"RANKING COMPLETO — {family}")
    return frame


### 8.4 Execução e acompanhamento dos testes
  
  A célula abaixo percorre todas as combinações, imprime cada candidato e mostra as tabelas completas ordenadas. Os checkpoints permitem continuar uma execução interrompida.


In [ ]:
base_results = search_base_models()
sarima_results = search_statistical_models("SARIMA")
sarimax_results = search_statistical_models("SARIMAX")


### 8.5 Seleção e ranking dos melhores modelos


In [ ]:
def best_valid_row(frame, filters=None):
    selected = frame.copy()
    if filters:
        for col, value in filters.items(): selected = selected[selected[col] == value]
    selected = sort_results(selected[selected["status"] == "ok"])
    if selected.empty: raise RuntimeError(f"Nenhum candidato válido para filtros={filters}")
    return selected.iloc[0]

def evaluate_base_on_test(best_row):
    pred = base_forecast(df_train["vendas"], len(df_test), best_row["params"])
    return pd.Series(pred, index=df_test.index), metrics(df_test["vendas"], pred)

def evaluate_statistical_on_test(best_row):
    fitted = fit_and_forecast(
        df_train, df_test, tuple(best_row["order"]), tuple(best_row["seasonal_order"]),
        best_row["trend"], list(best_row["exog_cols"])
    )
    return fitted, metrics(df_test["vendas"], fitted["forecast"])

best_base, best_sarima, best_sarimax = best_valid_row(base_results), best_valid_row(sarima_results), best_valid_row(sarimax_results)
base_forecast_test, base_score = evaluate_base_on_test(best_base)
sarima_final, sarima_score = evaluate_statistical_on_test(best_sarima)
sarimax_final, sarimax_score = evaluate_statistical_on_test(best_sarimax)

comparison_final = pd.DataFrame([
    {"modelo": "Base Model", "configuracao": best_base["params"], "s": best_base.get("s"), "exogenas": "nenhuma", "BIC": np.nan, "MAE_validacao": best_base["val_mae"], "DP_MAE_validacao": best_base["val_mae_std"], "MAE_teste": base_score["mae"], "MAPE_teste": base_score["mape"]},
    {"modelo": "SARIMA", "configuracao": {"order": best_sarima["order"], "seasonal_order": best_sarima["seasonal_order"], "trend": best_sarima["trend"]}, "s": best_sarima["s"], "exogenas": "nenhuma", "BIC": best_sarima["bic"], "MAE_validacao": best_sarima["val_mae"], "DP_MAE_validacao": best_sarima["val_mae_std"], "MAE_teste": sarima_score["mae"], "MAPE_teste": sarima_score["mape"]},
    {"modelo": "SARIMAX", "configuracao": {"order": best_sarimax["order"], "seasonal_order": best_sarimax["seasonal_order"], "trend": best_sarimax["trend"]}, "s": best_sarimax["s"], "exogenas": best_sarimax["exog_cols"], "BIC": best_sarimax["bic"], "MAE_validacao": best_sarimax["val_mae"], "DP_MAE_validacao": best_sarimax["val_mae_std"], "MAE_teste": sarimax_score["mae"], "MAPE_teste": sarimax_score["mape"]},
]).sort_values("MAE_teste")
display(comparison_final)

print("\nTOP 10 BASE MODELS"); display(sort_results(base_results).head(10))
print("\nTOP 10 SARIMA"); display(sort_results(sarima_results).head(10))
print("\nTOP 10 SARIMAX"); display(sort_results(sarimax_results).head(10))

plt.figure(figsize=(15, 6))
plt.plot(df_test.index, df_test["vendas"], color="black", linewidth=2.5, label="Real")
plt.plot(df_test.index, base_forecast_test, label="Melhor Base Model")
plt.plot(df_test.index, sarima_final["forecast"], label="Melhor SARIMA")
plt.plot(df_test.index, sarimax_final["forecast"], label="Melhor SARIMAX")
plt.title("Tarefa 3 — comparação final na escala original")
plt.ylabel("Vendas")
plt.legend()
plt.tight_layout()
plt.show()


### 8.6 Comparação das sazonalidades testadas


In [ ]:
def seasonality_ranking(frame, family_name):
    valid = frame[frame["status"] == "ok"].copy()
    if valid.empty: return pd.DataFrame()
    summary = valid.groupby("s", dropna=False).agg(
        melhor_mae=("val_mae", "min"), mae_mediano=("val_mae", "median"),
        melhor_bic=("bic", "min"), modelos_validos=("candidate_key", "count"),
        tempo_total=("elapsed_seconds", "sum"),
    ).reset_index().sort_values(["melhor_mae", "melhor_bic"])
    summary.insert(0, "posicao", np.arange(1, len(summary) + 1))
    print(f"\nSAZONALIDADES — {family_name}")
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        print(summary.to_string(index=False))
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    axes[0].plot(summary["s"], summary["melhor_mae"], marker="o", markersize=3)
    axes[0].set_title(f"{family_name}: s × melhor MAE"); axes[0].set_xlabel("s"); axes[0].set_ylabel("MAE")
    axes[1].plot(summary["s"], summary["melhor_bic"], marker="o", markersize=3)
    axes[1].set_title(f"{family_name}: s × melhor BIC"); axes[1].set_xlabel("s"); axes[1].set_ylabel("BIC")
    plt.tight_layout(); plt.show()
    return summary

sarima_seasonality = seasonality_ranking(sarima_results, "SARIMA")
sarimax_seasonality = seasonality_ranking(sarimax_results, "SARIMAX")


### 8.7 Análise dos resíduos dos melhores modelos


In [ ]:
def residual_diagnostics(result, model_name, alpha=0.05):
    residuals = pd.Series(result.resid).dropna()
    lag = min(20, max(1, len(residuals) // 5))
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes[0, 0].plot(residuals); axes[0, 0].axhline(0, color="black", linewidth=0.8); axes[0, 0].set_title(f"{model_name} — resíduos")
    axes[0, 1].hist(residuals, bins=20, color="#457b9d", edgecolor="white"); axes[0, 1].set_title(f"{model_name} — histograma")
    stats.probplot(residuals, dist="norm", plot=axes[1, 0]); axes[1, 0].set_title(f"{model_name} — Q-Q")
    plot_acf(residuals, lags=lag, ax=axes[1, 1], title=f"{model_name} — ACF dos resíduos")
    plt.tight_layout(); plt.show()
    lb = acorr_ljungbox(residuals, lags=[lag], return_df=True)
    display(lb)
    pvalue = float(lb["lb_pvalue"].iloc[0])
    print("Resíduos compatíveis com ruído branco." if pvalue > alpha else "Há autocorrelação significativa remanescente.")
    return {"media_residuos": float(residuals.mean()), "desvio_residuos": float(residuals.std()), "ljung_box_lag": lag, "ljung_box_p": pvalue}

diag_sarima = residual_diagnostics(sarima_final["result"], "Melhor SARIMA")
diag_sarimax = residual_diagnostics(sarimax_final["result"], "Melhor SARIMAX")


### 8.8 Consolidação final das três tarefas

Nesta etapa, os modelos vencedores encontrados na Tarefa 3 são usados para consolidar as comparações da dummy de dezembro e dos cenários com e sem feriado solicitados nas Tarefas 1 e 2.


In [ ]:
task1_optimized_specs = [
    ("Melhor SARIMA", best_sarima),
    ("Melhor SARIMAX temperatura+feriado", best_valid_row(sarimax_results, {"exog_key": "temperatura_feriado"})),
    ("Melhor SARIMAX temperatura+feriado+dezembro", best_valid_row(sarimax_results, {"exog_key": "temperatura_feriado_dezembro"})),
]
optimized_rows = []
for name, row in task1_optimized_specs:
    fitted, score = evaluate_statistical_on_test(row)
    optimized_rows.append({"modelo": name, "order": row["order"], "seasonal_order": row["seasonal_order"], "s": row["s"], "exogenas": row["exog_cols"], "BIC": row["bic"], "MAE": score["mae"], "MAPE": score["mape"]})
task1_optimized = pd.DataFrame(optimized_rows).sort_values(["MAE", "BIC"])
display(task1_optimized)

best_task2 = task1_optimized_specs[1][1]
task2_exog_cols = list(best_task2["exog_cols"])
future_no_holiday_opt = pd.DataFrame({"temperatura": 22.0, "feriado": 0}, index=future_index)
future_holiday_opt = pd.DataFrame({"temperatura": 22.0, "feriado": 1}, index=future_index)
x_history = df[task2_exog_cols].astype(float).copy()
x_no = future_no_holiday_opt[task2_exog_cols].astype(float).copy()
x_yes = future_holiday_opt[task2_exog_cols].astype(float).copy()
continuous = [c for c in ["temperatura"] if c in task2_exog_cols]
if continuous:
    best_scenario_scaler = StandardScaler()
    x_history.loc[:, continuous] = best_scenario_scaler.fit_transform(x_history[continuous])
    x_no.loc[:, continuous] = best_scenario_scaler.transform(x_no[continuous])
    x_yes.loc[:, continuous] = best_scenario_scaler.transform(x_yes[continuous])
best_scenario_result, _, _, _ = fit_sarimax_model(df["vendas"], x_history, tuple(best_task2["order"]), tuple(best_task2["seasonal_order"]), best_task2["trend"])
pred_no = best_scenario_result.get_forecast(steps=4, exog=x_no).predicted_mean
pred_yes = best_scenario_result.get_forecast(steps=4, exog=x_yes).predicted_mean
pred_no.index = pred_yes.index = future_index
task2_optimized = pd.DataFrame({"sem_feriado": pred_no, "com_feriado": pred_yes})
task2_optimized["diferenca"] = task2_optimized["com_feriado"] - task2_optimized["sem_feriado"]
display(task2_optimized)
print(f"Diferença média com o melhor SARIMAX: {task2_optimized['diferenca'].mean():.2f} unidades")
